In [4]:
import cv2
import numpy as np
import os
from deepface import DeepFace
import mediapipe as mp
import time
import random


DATABASE_FILE = "student_embeddings_deepface.npz"
FRAME_SKIP_RATE = 5
SIMILARITY_THRESHOLD = 0.40

def load_database():
    if os.path.exists(DATABASE_FILE):
        data = np.load(DATABASE_FILE, allow_pickle=True)
        return {key: data[key] for key in data.files}
    return {}

def save_database(database):
    np.savez_compressed(DATABASE_FILE, **database)
    print(f"💾 Database saved! Total students: {len(database)}")

print("✅ System Ready. DeepFace loaded.")

ValueError: You have tensorflow 2.19.1 and this requires tf-keras package. Please run `pip install tf-keras` or downgrade your tensorflow.

In [2]:
data = np.load("student_embeddings_deepface.npz")
print(data.files)

['2702367170', '2702238110']


In [7]:
# --- CELL 2: ENROLLMENT (DeepFace) ---
# Run this to add a new user to the database

student_id = input("Enter Student ID to Enroll (e.g., '12345'): ")
database = load_database()

captured_encodings = []
REQUIRED_SAMPLES = 5

# Change index to 1 if you are using an external USB camera
cap = cv2.VideoCapture(0) 

print(f"\n🎥 ENROLLMENT FOR: {student_id}")
print(f"👉 Press 's' to capture ({REQUIRED_SAMPLES} needed). 'q' to quit.")

while True:
    ret, frame = cap.read()
    if not ret: break
    
    display_frame = frame.copy()
    cv2.putText(display_frame, f"Samples: {len(captured_encodings)}/{REQUIRED_SAMPLES}", (20, 50), 
                cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 0), 2)
    cv2.imshow("Enrollment", display_frame)
    
    key = cv2.waitKey(1) & 0xFF
    if key == ord('s'):
        try:
            # Generate Embedding using DeepFace (Facenet model)
            # enforce_detection=True ensures we only save valid faces
            results = DeepFace.represent(
                img_path = frame, 
                model_name = "Facenet", 
                enforce_detection = True,
                detector_backend = "opencv"
            )
            
            if results:
                embedding = results[0]["embedding"]
                captured_encodings.append(embedding)
                print(f"✅ Sample {len(captured_encodings)} captured!")
                
                # Visual Flash effect
                cv2.rectangle(display_frame, (0,0), (frame.shape[1], frame.shape[0]), (255,255,255), 5)
                cv2.imshow("Enrollment", display_frame)
                cv2.waitKey(50)
        except:
            print("⚠️ Face not detected. Look at the camera and try again.")

    if len(captured_encodings) >= REQUIRED_SAMPLES:
        print("\n✨ Collection Complete!")
        break
    if key == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

# Save Average Data
if len(captured_encodings) == REQUIRED_SAMPLES:
    final_embedding = np.mean(captured_encodings, axis=0)
    database[student_id] = final_embedding
    save_database(database)
    print(f"✅ Student {student_id} successfully enrolled.")


🎥 ENROLLMENT FOR: 2702238110
👉 Press 's' to capture (5 needed). 'q' to quit.
✅ Sample 1 captured!
⚠️ Face not detected. Look at the camera and try again.
✅ Sample 2 captured!
⚠️ Face not detected. Look at the camera and try again.
⚠️ Face not detected. Look at the camera and try again.
✅ Sample 3 captured!
⚠️ Face not detected. Look at the camera and try again.
⚠️ Face not detected. Look at the camera and try again.
✅ Sample 4 captured!
✅ Sample 5 captured!

✨ Collection Complete!
💾 Database saved! Total students: 2
✅ Student 2702238110 successfully enrolled.


In [ ]:
# ===== CONFIGURATION =====
DATABASE_FILE = "student_embeddings_deepface.npz"
SIMILARITY_THRESHOLD = 0.20
CAM_ID = 0

BLINK_THRESH = 0.21
TURN_THRESH = 20
CHALLENGE_TIMEOUT = 5.0

HOLD_FRAMES = 50
BLINK_MIN_FRAMES = 6

confidence = 0.0

# ===== MEDIAPIPE SETUP =====
mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(
    max_num_faces=1,
    refine_landmarks=True,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

face_3d = np.array([
    [0.0, 0.0, 0.0],
    [0.0, -330.0, -65.0],
    [-225.0, 170.0, -135.0],
    [225.0, 170.0, -135.0],
    [-150.0, -150.0, -125.0],
    [150.0, -150.0, -125.0]
], dtype=np.float64)

FACE_2D_IDX = [1, 152, 33, 263, 61, 291]
LEFT_EYE = [33, 160, 158, 133, 153, 144]
RIGHT_EYE = [263, 387, 385, 362, 380, 373]

# ===== HELPERS =====
def load_database():
    if os.path.exists(DATABASE_FILE):
        data = np.load(DATABASE_FILE, allow_pickle=True)
        return {key: data[key] for key in data.files}
    return {}

def calculate_ear(landmarks, w, h, idxs):
    pts = np.array([(landmarks[i].x * w, landmarks[i].y * h) for i in idxs])
    v1 = np.linalg.norm(pts[1] - pts[5])
    v2 = np.linalg.norm(pts[2] - pts[4])
    h1 = np.linalg.norm(pts[0] - pts[3])
    if h1 == 0: return 0
    return (v1 + v2) / (2.0 * h1)

def get_head_pose(landmarks, img_w, img_h):
    face_2d = []
    for idx in FACE_2D_IDX:
        lm = landmarks[idx]
        face_2d.append([lm.x * img_w, lm.y * img_h])
    
    face_2d = np.array(face_2d, dtype=np.float64)
    focal_length = 1 * img_w
    cam_matrix = np.array([[focal_length, 0, img_h / 2], [0, focal_length, img_w / 2], [0, 0, 1]])
    dist_matrix = np.zeros((4, 1), dtype=np.float64)
    
    success, rot_vec, trans_vec = cv2.solvePnP(face_3d, face_2d, cam_matrix, dist_matrix)
    rmat, jac = cv2.Rodrigues(rot_vec)
    angles, mtxR, mtxQ, Qx, Qy, Qz = cv2.RQDecomp3x3(rmat)
    
    return angles[0] * 360, angles[1] * 360, angles[2] * 360

def draw_progress_bar(img, progress, max_val, x, y, w, h, color=(0, 255, 0)):
    cv2.rectangle(img, (x, y), (x + w, y + h), (50, 50, 50), -1)
    ratio = min(progress / max_val, 1.0)
    fill_w = int(w * ratio)
    cv2.rectangle(img, (x, y), (x + fill_w, y + h), color, -1)
    cv2.rectangle(img, (x, y), (x + w, y + h), (255, 255, 255), 1)

# ===== MAIN EXECUTION =====
claimed_id = input("Enter Claimed User ID: ")
database = load_database()
ref_emb = database.get(claimed_id)

if ref_emb is None:
    print("❌ User not found.")
else:
    print("✅ Starting Natural Challenge Liveness...")
    cap = cv2.VideoCapture(CAM_ID)
    
    CHALLENGES = ["LOOK LEFT", "LOOK RIGHT", "BLINK"]
    target_challenge = None
    challenge_start_time = 0
    challenges_passed = 0
    REQUIRED_PASSES = 3
    
    hold_counter = 0
    blink_close_frames = 0
    
    status_msg = "CENTER FACE"
    box_color = (255, 255, 0)
    verified = False

    while True:
        ret, frame = cap.read()
        if not ret: break
        
        frame = cv2.flip(frame, 1)
        h, w, _ = frame.shape
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = face_mesh.process(rgb)
        
        if results.multi_face_landmarks:
            lm = results.multi_face_landmarks[0].landmark
            
            pitch, yaw, roll = get_head_pose(lm, w, h)
            left_ear = calculate_ear(lm, w, h, LEFT_EYE)
            right_ear = calculate_ear(lm, w, h, RIGHT_EYE)
            avg_ear = (left_ear + right_ear) / 2.0
            
            if verified:
                status_msg = f"ACCESS GRANTED: {claimed_id}"
                box_color = (0, 255, 0)
            
            elif target_challenge is None:
                target_challenge = random.choice(CHALLENGES)
                challenge_start_time = time.time()
                hold_counter = 0
                blink_close_frames = 0
                status_msg = f"DO THIS: {target_challenge}"
                box_color = (0, 255, 255)
            
            else:
                time_elapsed = time.time() - challenge_start_time
                status_msg = f"CMD: {target_challenge}"
                passed_now = False
                
                if target_challenge == "LOOK LEFT":
                    if yaw < -TURN_THRESH:
                        hold_counter += 1
                    else:
                        hold_counter = max(0, hold_counter - 2)
                    
                    if hold_counter >= HOLD_FRAMES: passed_now = True
                    draw_progress_bar(frame, hold_counter, HOLD_FRAMES, 20, 100, 200, 20)

                elif target_challenge == "LOOK RIGHT":
                    if yaw > TURN_THRESH:
                        hold_counter += 1
                    else:
                        hold_counter = max(0, hold_counter - 2)
                        
                    if hold_counter >= HOLD_FRAMES: passed_now = True
                    draw_progress_bar(frame, hold_counter, HOLD_FRAMES, 20, 100, 200, 20)

                elif target_challenge == "BLINK":
                    if avg_ear < BLINK_THRESH:
                        blink_close_frames += 1
                    else:
                        if blink_close_frames >= BLINK_MIN_FRAMES:
                            passed_now = True
                        blink_close_frames = 0
                
                if passed_now:
                    challenges_passed += 1
                    target_challenge = None
                    hold_counter = 0
                    status_msg = "GOOD!"
                    box_color = (0, 255, 0)
                    
                    cv2.rectangle(frame, (0,0), (w,h), (0,255,0), 10)
                    cv2.imshow("Secure Attendance", frame)
                    cv2.waitKey(250)
                    
                    if challenges_passed >= REQUIRED_PASSES:
                        process_start = time.time()
                        while time.time() - process_start < 2.0:
                            ret_p, frame_p = cap.read()
                            if not ret_p: break
                            frame_p = cv2.flip(frame_p, 1)
                            cv2.putText(frame_p, "VERIFYING BIOMETRICS...", (50, h//2), 
                                        cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
                            cv2.imshow("Secure Attendance", frame_p)
                            cv2.waitKey(1)

                        try:
                            df_res = DeepFace.represent(frame, model_name="Facenet", enforce_detection=False)
                            if df_res:
                                live_emb = np.array(df_res[0]["embedding"])
                                a = np.array(ref_emb)
                                b = live_emb

                                # --- Cosine similarity ---
                                similarity = np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

                                # Convert your existing distance metric
                                dist = 1 - similarity

                                # Confidence score = similarity
                                confidence = similarity  # between 0 and 1

                                print(f"Similarity: {similarity:.4f}")
                                print(f"Distance: {dist:.4f}")
                                print(f"Confidence: {confidence*100:.2f}%")
                                
                                if confidence >= 0.80:  # 80% similarity threshold
                                    verified = True
                                else:
                                    status_msg = "FACE MISMATCH"
                                    box_color = (0, 0, 255)
                                    target_challenge = "FAILED"
                        except: pass
                
                if time_elapsed > CHALLENGE_TIMEOUT:
                    status_msg = "TIMEOUT - FAKE"
                    box_color = (0, 0, 255)
                    challenges_passed = 0
                    target_challenge = None

            nose_x, nose_y = int(lm[1].x * w), int(lm[1].y * h)
            cv2.arrowedLine(frame, (nose_x, nose_y), 
                           (nose_x + int(yaw * 2), nose_y + int(pitch * 2)), (255, 0, 0), 2)
            
            x_min = int(min([l.x for l in lm]) * w)
            y_min = int(min([l.y for l in lm]) * h)
            x_max = int(max([l.x for l in lm]) * w)
            y_max = int(max([l.y for l in lm]) * h)
            cv2.rectangle(frame, (x_min, y_min), (x_max, y_max), box_color, 2)
            
            cv2.putText(frame, status_msg, (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, box_color, 3)
            cv2.putText(frame, f"Pass: {challenges_passed}/{REQUIRED_PASSES}", (20, h-30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (200,200,200), 2)
            cv2.putText(frame, f"Conf: {confidence*100:.1f}%", 
            (20, h - 60), cv2.FONT_HERSHEY_SIMPLEX, 
            0.7, (0,255,0) if verified else (0,0,255), 2)


        else:
            status_msg = "LOOK AT CAMERA"
            box_color = (100, 100, 100)
            target_challenge = None
            challenges_passed = 0
            verified = False
            hold_counter = 0
        
        cv2.imshow("Secure Attendance", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'): break

    cap.release()
    cv2.destroyAllWindows()

✅ Starting Natural Challenge Liveness...
